## Setup

> **Kernel:** run this notebook in the **`.conda`** env (`/Users/diyagoel/slp-diss/.conda/bin/python`).
> It provides metrics 2/3/4/7 (F0 RMSE, MCD, WER, PPG-KL) and shells out to the `genaid`
> env for metrics 5/6/8 (accent-ID, accent CS, speaker SECS). Metric 1 (UTMOS) lives only
> in the root `.venv`, so eval 1 runs it as a subprocess too. See `README.md`.

In [ ]:
# imports
import os, sys, json, subprocess, zipfile
from pathlib import Path

import numpy as np
import pandas as pd

# evaluation_functions.py lives next to this notebook (run with cwd = SOTA_models_experiments).
sys.path.insert(0, str(Path.cwd()))
import evaluation_functions as ef

# UTMOS (metric 1) is only installed in the root .venv; eval 1 calls this interpreter.
VENV_PYTHON = "/Users/diyagoel/slp-diss/.venv/bin/python"

pd.set_option("display.width", 160)
pd.set_option("display.float_format", lambda x: f"{x:.4f}")

In [ ]:
# load data
# Grid + paths come from eval_config.py — the SAME module synthesis_driver.py uses, so the
# synthesised file layout always matches what we score here.
import eval_config as cfg
from eval_config import UTTERANCES, ACCENT_SPEAKERS, MODELS, L2_ARCTIC, SOTA, ref_path, synth_path

# extract just the reference wavs we need (10 utts x 16 speakers) from the speaker zips.
for speakers in ACCENT_SPEAKERS.values():
    for spk in speakers:
        cfg.ensure_wavs(spk, list(UTTERANCES))
print("references ready under", L2_ARCTIC)

# --- build the evaluation manifest: one row per (model, accent, speaker, utterance) ---
rows = []
for model in MODELS:
    for cell in cfg.grid():
        rows.append({
            "model": model, **{k: cell[k] for k in ("accent", "speaker", "utt_id", "text")},
            "synth": str(synth_path(model, cell["accent"], cell["speaker"], cell["utt_id"])),
            "ref":   str(ref_path(cell["speaker"], cell["utt_id"])),
        })
manifest = pd.DataFrame(rows)

# coverage: we can only score clips that have been synthesised AND have a reference.
manifest["synth_exists"] = manifest["synth"].map(lambda p: Path(p).exists())
manifest["ref_exists"]   = manifest["ref"].map(lambda p: Path(p).exists())
manifest["evaluable"]    = manifest["synth_exists"] & manifest["ref_exists"]

print(f"\nmanifest: {len(manifest)} rows "
      f"({len(MODELS)} models x {len(ACCENT_SPEAKERS)} accents x 4 speakers x {len(UTTERANCES)} utts)")
print(manifest.groupby("model")[["synth_exists", "ref_exists", "evaluable"]].sum())

# `df` is the working set the eval cells run over (skips clips not yet synthesised).
df = manifest[manifest["evaluable"]].reset_index(drop=True).copy()
print(f"\nevaluable clips: {len(df)}")
df.head()

**Data layout**

- **Reference (natural):** `Datasets/L2-ARCTIC/{SPEAKER}/wav/{utt_id}.wav` — the L2 speaker's own accented recording of the utterance (auto-extracted from the speaker zips above).
- **Synthesised:** `SOTA_models_experiments/{model}/outputs/{model}/{accent}/{speaker}/{utt_id}.wav`.

Each synthesised clip is scored against its **paired natural reference** (same speaker + same utterance).

**Matrix:** 5 models × 4 accents × 4 speakers × 10 utterances = **800 clips**.

| accent | L2-ARCTIC speakers (L1) |
|---|---|
| Arabic | ABA, SKA, YBAA, ZHAA |
| Indian | ASI, RRBI, SVBI, TNI (Hindi) |
| Korean | HJK, HKK, YDCK, YKWK |
| Vietnamese | HQTV, PNV, THV, TLV |

The eval cells run only over clips already on disk (`df`), so you can run the notebook incrementally as synthesis completes.

## Evaluations

In [ ]:
# shared helper: apply a pairwise metric fn(synth, ref) over every row of df,
# isolating per-clip failures so one bad file doesn't abort the whole sweep.
def per_pair(fn, label, second="ref"):
    vals = []
    for _, r in df.iterrows():
        try:
            vals.append(float(fn(r["synth"], r[second])))
        except Exception as e:
            print(f"  [{label}] {r['model']}/{r['speaker']}/{r['utt_id']}: {e}")
            vals.append(np.nan)
    return vals

In [ ]:
# eval 1 — UTMOS (naturalness MOS, higher is better)
# utmosv2 only exists in the root .venv, so run it there as a subprocess. It predicts
# over a *directory*, so we group synth clips by parent dir and map scores back by path.
def utmos_venv(wav_paths):
    scores = {}
    by_dir = {}
    for p in wav_paths:
        rp = str(Path(p).resolve())
        by_dir.setdefault(str(Path(rp).parent), {})[Path(rp).name] = rp
    script = (
        "import sys, json, utmosv2;"
        "m = utmosv2.create_model(pretrained=True);"
        "print('@@@' + json.dumps(m.predict(input_dir=sys.argv[1])))"
    )
    for d, namemap in by_dir.items():
        out = subprocess.run([VENV_PYTHON, "-c", script, d],
                             capture_output=True, text=True, check=True)
        payload = next(l for l in out.stdout.splitlines() if l.startswith("@@@"))[3:]
        for rec in json.loads(payload):
            name = Path(rec["file_path"]).name
            if name in namemap:
                scores[namemap[name]] = rec["predicted_mos"]
    return scores

if len(df):
    mos = utmos_venv(df["synth"].tolist())
    df["utmos"] = df["synth"].map(lambda p: mos.get(str(Path(p).resolve()), np.nan))
    display(df[["model", "accent", "speaker", "utt_id", "utmos"]].head())
else:
    print("no evaluable clips yet")

In [ ]:
# eval 2 — F0 RMSE (mel-scaled pitch distance vs natural reference, lower is better)
if len(df):
    df["f0_rmse"] = per_pair(ef.f0_rmse, "f0_rmse")
    display(df[["model", "accent", "speaker", "utt_id", "f0_rmse"]].head())
else:
    print("no evaluable clips yet")

In [ ]:
# eval 3 — MCD (mel-cepstral distortion / spectral envelope, dB, lower is better)
if len(df):
    df["mcd"] = per_pair(ef.mcd, "mcd")
    display(df[["model", "accent", "speaker", "utt_id", "mcd"]].head())
else:
    print("no evaluable clips yet")

In [ ]:
# eval 4 — WER (intelligibility via Whisper + jiwer, lower is better)
# Scored against the prompt text, not the reference audio (second="text").
if len(df):
    df["wer"] = per_pair(ef.wer, "wer", second="text")
    display(df[["model", "accent", "speaker", "utt_id", "wer"]].head())
else:
    print("no evaluable clips yet")

In [ ]:
# eval 5 — accent-ID (GenAID, batched in the genaid env)
# CAVEAT: GenAID's classes are *native* English accents. Of our four L2 accents only
# "Indian" (southasian) has a class, so `aid_correct` is only meaningful for Indian.
# We keep the raw predicted label per clip for inspection; the real accent metric for
# L2 accents is eval 6 (cosine sim to the speaker's own accented recording).
if len(df):
    preds = ef.predict_accent_genaid(df["synth"].tolist())
    pred_by_wav = {p["wav"]: p["pred_accent"] for p in preds}

    def lookup_accent(p):
        raw = pred_by_wav.get(p) or pred_by_wav.get(str(Path(p).resolve()))
        return ef.GENAID_TO_VCTK.get(raw, raw)

    df["aid_pred"]    = df["synth"].map(lookup_accent)
    df["aid_correct"] = df["aid_pred"] == df["accent"]
    print("overall accent-ID accuracy:", df["aid_correct"].mean())
    print("\naccuracy by target accent:")
    print(df.groupby("accent")["aid_correct"].mean())
    print("\npredicted-label distribution:")
    print(df.groupby(["accent", "aid_pred"]).size())
else:
    print("no evaluable clips yet")

In [ ]:
# eval 6 — accent-embedding cosine similarity (GenAID, higher is better)
# Synth vs the speaker's own natural L2 recording — taxonomy-free, so this is the
# meaningful accent metric for our L2 accents. Batched in the genaid env.
if len(df):
    _, sims = ef.cs_accent(df["synth"].tolist(), df["ref"].tolist(), return_per_pair=True)
    df["cs_accent"] = sims
    display(df[["model", "accent", "speaker", "utt_id", "cs_accent"]].head())
else:
    print("no evaluable clips yet")

In [ ]:
# eval 7 — PPG-KL (segmental pronunciation fidelity vs reference, lower is better)
if len(df):
    df["ppg_kl"] = per_pair(ef.ppg_kl, "ppg_kl")
    display(df[["model", "accent", "speaker", "utt_id", "ppg_kl"]].head())
else:
    print("no evaluable clips yet")

In [ ]:
# eval 8 — speaker similarity / SECS (ECAPA-TDNN cosine sim, higher is better)
# Synth vs the speaker's own natural recording. Batched in the genaid env.
if len(df):
    _, ssims = ef.speaker_similarity(df["synth"].tolist(), df["ref"].tolist(), return_per_pair=True)
    df["speaker_sim"] = ssims
    display(df[["model", "accent", "speaker", "utt_id", "speaker_sim"]].head())
else:
    print("no evaluable clips yet")

## Analysis

Metric directions — **higher is better:** `utmos`, `cs_accent`, `speaker_sim`, `aid_correct`;
**lower is better:** `f0_rmse`, `mcd`, `wer`, `ppg_kl`.

In [ ]:
# collect whichever metric columns have been computed
METRIC_COLS = ["utmos", "f0_rmse", "mcd", "wer", "aid_correct", "cs_accent", "ppg_kl", "speaker_sim"]
present = [c for c in METRIC_COLS if c in df.columns]

# persist the per-clip results so analysis survives a kernel restart
out_csv = SOTA / "evaluation_results.csv"
df.to_csv(out_csv, index=False)
print(f"wrote {len(df)} rows x {len(present)} metrics -> {out_csv}")

# per-model summary (mean over all clips)
summary = df.groupby("model")[present].mean(numeric_only=True)
display(summary)

In [ ]:
# per-model x accent breakdown (does a model handle some accents better than others?)
by_accent = df.groupby(["model", "accent"])[present].mean(numeric_only=True)
display(by_accent)

# per-accent means across all models (which accents are hardest overall?)
display(df.groupby("accent")[present].mean(numeric_only=True))